# Sofifa Scraping

In [5]:
import csv
import time
import random
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright
import sys
import asyncio
import os
from playwright_stealth import Stealth

# Force Windows to use the Proactor Event Loop for subprocess support
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ==========================================
# PHASE 1: CONFIGURATION
# ==========================================
# REPLACE THIS with your massive URL containing all 46 columns
BASE_URL = "https://sofifa.com/players?&showCol%5B%5D=pi&showCol%5B%5D=ae&showCol%5B%5D=by&showCol%5B%5D=hi&showCol%5B%5D=pf&showCol%5B%5D=oa&showCol%5B%5D=pt&showCol%5B%5D=bp&showCol%5B%5D=gu&showCol%5B%5D=vl&showCol%5B%5D=wg&showCol%5B%5D=ta&showCol%5B%5D=cr&showCol%5B%5D=fi&showCol%5B%5D=he&showCol%5B%5D=sh&showCol%5B%5D=vo&showCol%5B%5D=ts&showCol%5B%5D=dr&showCol%5B%5D=cu&showCol%5B%5D=fr&showCol%5B%5D=lo&showCol%5B%5D=bl&showCol%5B%5D=to&showCol%5B%5D=ac&showCol%5B%5D=sp&showCol%5B%5D=ag&showCol%5B%5D=tp&showCol%5B%5D=so&showCol%5B%5D=ju&showCol%5B%5D=st&showCol%5B%5D=sr&showCol%5B%5D=ln&showCol%5B%5D=te&showCol%5B%5D=vi&showCol%5B%5D=pe&showCol%5B%5D=td&showCol%5B%5D=ma&showCol%5B%5D=sa&showCol%5B%5D=sl&showCol%5B%5D=tg&showCol%5B%5D=gd&showCol%5B%5D=gh&showCol%5B%5D=gc&showCol%5B%5D=gp&showCol%5B%5D=gr"
CSV_FILENAME = "data/sofifa/newdata/sofifa_players.csv"
TOTAL_PLAYERS = 400000 
PLAYERS_PER_PAGE = 60

# ==========================================
# PHASE 2: DYNAMIC EXTRACTION LOGIC
# ==========================================
def extract_headers_from_html(soup):
    header_row = soup.select_one("table thead tr")
    if not header_row:
        return []
    
    headers = []
    for th in header_row.find_all("th"):
        text = th.get_text(strip=True)
        # SoFifa's first column (the avatar picture) has no text in the header.
        # We dynamically rename this header to "ID" for our CSV.
        if not text and 'col-avatar' in th.get('class', []):
            headers.append("ID")
        elif not text:
            headers.append("Unknown")
        else:
            headers.append(text)
            
    return headers

def extract_rows_from_html(soup, limit=None):
    player_data = []
    rows = soup.select("table tbody tr")
        
    for row in rows:
        try:
            cols = row.find_all("td")
            
            # Bulletproof ad blocker (ads have very few columns)
            if len(cols) < 5:
                continue
                
            row_values = []
            
            for td in cols:
                classes = td.get('class', [])
                
                # 1. NAME EXTRACTION (Strictly Isolated)
                if 'col-name' in classes:
                    links = td.find_all("a", href=lambda h: h and "/player/" in h)
                    name_text = ""
                    for a in links:
                        # Find the hyperlink that actually has the text of the name, not the avatar image
                        if not a.find("img") and a.get_text(strip=True):
                            name_text = a.get_text(strip=True)
                            break
                            
                    if not name_text:
                        # Fallback just in case
                        name_div = td.select_one(".bp3-text-overflow-ellipsis")
                        name_text = name_div.get_text(strip=True) if name_div else td.get_text(separator=" ", strip=True)
                        
                    row_values.append(name_text)
                        
                # 2. AVATAR/HIDDEN ID EXTRACTION
                elif 'col-avatar' in classes:
                    link = td.find("a", href=lambda h: h and "/player/" in h)
                    row_values.append(link['href'].split('/')[2] if link else "")
                        
                # 3. ALL OTHER STATS (Including the real ID column)
                else:
                    row_values.append(td.get_text(separator=" ", strip=True))
                    
            player_data.append(row_values)
            
            if limit and len(player_data) == limit:
                break
            
        except Exception as e:
            continue
            
    return player_data

# ==========================================
# PHASE 3: THE TURBO BROWSER THREAD
# ==========================================
def _run_playwright_pipeline(is_test=True):
    os.makedirs(os.path.dirname(CSV_FILENAME), exist_ok=True)

    with sync_playwright() as p:
        # 1. Setup Persistent Profile Folder
        profile_path = os.path.join(os.getcwd(), "data", "sofifa_profile")
        os.makedirs(profile_path, exist_ok=True)
        
        # 2. Launch actual Chrome with saved cookies
        context = p.chromium.launch_persistent_context(
            user_data_dir=profile_path,
            channel="chrome", 
            headless=False,
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        
        # 3. Resource Blocker
        def block_heavy_resources(route):
            if route.request.resource_type in ["image", "stylesheet", "font", "media"]:
                route.abort()
            else:
                route.continue_()
                
        context.route("**/*", block_heavy_resources)
        print("Turbo mode activated (Images/CSS blocked for extreme speed).\n")

        # 4. Grab the default tab and apply Stealth
        page = context.pages[0]
        stealth = Stealth()
        stealth.apply_stealth_sync(page)
        
        print("Launching persistent stealth browser to solve Cloudflare challenge...")
        
        try:
            # 3. Now it is safe to navigate
            page.goto(f"{BASE_URL}&offset=0")
            page.wait_for_selector("table tbody tr", timeout=30000)
            
            # Dynamic wait
            page.wait_for_function(
                "() => document.querySelectorAll('table tbody tr td').length > 100",
                timeout=15000
            )
            print("Challenge passed! Table loaded.")
            
        except Exception as e:
            print(f"Failed to bypass Cloudflare. Error: {e}")
            context.close()
            return

        # ==========================================
        # TEST BATCH LOGIC
        # ==========================================
        if is_test:
            print("--- RUNNING 10-PLAYER VALIDATION TEST ---")
            
            # # Reload page with Turbo Mode active
            # page.goto(f"{BASE_URL}&offset=0")
            # page.wait_for_selector("table tbody tr") 
            
            soup = BeautifulSoup(page.content(), "html.parser")
            columns = extract_headers_from_html(soup)
            players = extract_rows_from_html(soup, limit=10)

            print(f"Successfully Fetched! Extracted {len(columns)} Columns.")
            print("-" * 50)
            for i, player in enumerate(players):
                player_dict = dict(zip(columns, player))
                print(f"Player {i+1}: {player_dict}")
            print("-" * 50)
            print("Test complete. Clear to run production scrape.\n")
            
        # ==========================================
        # PRODUCTION SCRAPE LOGIC
        # ==========================================
        else:
            # 1. Check if we have an existing file to resume from
            start_offset = 0
            if os.path.exists(CSV_FILENAME):
                with open(CSV_FILENAME, "r", encoding="utf-8") as f:
                    # Count lines minus 1 for the header
                    existing_rows = sum(1 for line in f) - 1
                    if existing_rows > 0:
                        # Round down to the nearest multiple of 60 to ensure clean pagination
                        start_offset = (existing_rows // PLAYERS_PER_PAGE) * PLAYERS_PER_PAGE

            print(f"--- STARTING PRODUCTION SCRAPE ---")
            if start_offset > 0:
                print(f"[Resume Mode] Found {existing_rows} existing players. Resuming from offset {start_offset}...\n")
            else:
                print("[New Run] No existing data found. Starting from scratch...\n")

                # Initialize fresh CSV file with headers
                page.goto(f"{BASE_URL}&offset=0")
                page.wait_for_selector("table tbody tr")
                soup = BeautifulSoup(page.content(), "html.parser")
                headers = extract_headers_from_html(soup)
                
                with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
                    writer = csv.writer(file)
                    writer.writerow(headers)
                
            # 2. Master Loop (Starts at start_offset)
            with tqdm(total=TOTAL_PLAYERS, initial=start_offset, desc="Scraping SoFifa", unit=" players") as pbar:
                for offset in range(start_offset, TOTAL_PLAYERS, PLAYERS_PER_PAGE):
                    url = f"{BASE_URL}&offset={offset}"
                    
                    success = False
                    for attempt in range(3):
                        try:
                            # Only navigate if it's not the first load (or if we are resuming)
                            if offset != 0 or start_offset > 0 or attempt > 0:
                                page.goto(url)
                                page.wait_for_selector("table tbody tr", timeout=30000)
                                
                                # Dynamic wait: Checks if the whole table has populated
                                page.wait_for_function(
                                    "() => document.querySelectorAll('table tbody tr td').length > 100",
                                    timeout=15000
                                )
                                
                            soup = BeautifulSoup(page.content(), "html.parser")
                            players = extract_rows_from_html(soup)
                            
                            # Append strictly to CSV (mode="a")
                            with open(CSV_FILENAME, mode="a", newline="", encoding="utf-8") as file:
                                writer = csv.writer(file)
                                writer.writerows(players)
                            
                            pbar.update(len(players))
                            success = True
                            
                            if len(players) == 0:
                                print("\n[Notice] No more players found. Database exhausted.")
                                context.close()
                                return
                            break 
                            
                        except Exception as e:
                            print(f"\n[Error on offset {offset}]. Attempt {attempt + 1}/3.")
                            print(f"Details: {str(e)}")  # <--- This will tell us exactly what failed
                            time.sleep(5)
                    
                    if not success:
                        print(f"\n[Fatal] Failed to fetch offset {offset} after 3 attempts. Stopping script to prevent data gaps.")
                        break

                    # Polite delay
                    time.sleep(random.uniform(1.5, 3.0))

            print(f"\nScraping complete! Data safely saved to {CSV_FILENAME}")

        # Safely shut down Chromium
        context.close()

# ==========================================
# EXECUTION CONTROLS
# ==========================================
def run_test():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=True).result()

def run_production():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=False).result()

C:\Users\ashwy\AppData\Local\Temp\ipykernel_17536\2740390964.py:15: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\ashwy\AppData\Local\Temp\ipykernel_17536\2740390964.py:15: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


In [6]:
run_test()

Turbo mode activated (Images/CSS blocked for extreme speed).

Launching persistent stealth browser to solve Cloudflare challenge...
Challenge passed! Table loaded.
--- RUNNING 10-PLAYER VALIDATION TEST ---
Successfully Fetched! Extracted 50 Columns.
--------------------------------------------------
Player 1: {'Unknown': '', 'Name': 'K. Coulibaly CB CDM CM', 'Age': '18', 'Overall rating': '71 +2', 'Potential': '85 +3', 'Team & Contract': 'SV Werder Bremen 2025 ~ 2029', 'ID': '77636', 'Birth year': '2007', 'Height': '191cm 6\'3"', 'foot': 'Left', 'Best position': 'CB', 'Growth': '14', 'Value': '€4.1M', 'Wage': '€11K', 'Total attacking': '248', 'Crossing': '45 +2', 'Finishing': '35 +2', 'Heading accuracy': '65 +3', 'Short passing': '70 +2', 'Volleys': '33', 'Total skill': '238', 'Dribbling': '50 +3', 'Curve': '38 +4', 'FK Accuracy': '25', 'Long passing': '66 +3', 'Ball control': '59 +1', 'Total movement': '322', 'Acceleration': '61', 'Sprint speed': '68', 'Agility': '62', 'Total power': 

In [ ]:
run_production()

Turbo mode activated (Images/CSS blocked for extreme speed).

Launching persistent stealth browser to solve Cloudflare challenge...
Challenge passed! Table loaded.
--- STARTING PRODUCTION SCRAPE ---
[Resume Mode] Found 975 existing players. Resuming from offset 960...



Scraping SoFifa:   0%|          | 970/400000 [00:22<253:14:27,  2.28s/ players]

In [3]:
import csv
import time
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import asyncio
from playwright.sync_api import sync_playwright
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ==========================================
# PHASE 1: CONFIGURATION
# ==========================================
# Exact URL, no offset parameters needed for leagues
BASE_URL = "https://sofifa.com/leagues"
CSV_FILENAME = "data/sofifa/newdata/sofifa_raw_leagues.csv"

# ==========================================
# PHASE 2: DYNAMIC EXTRACTION LOGIC
# ==========================================
def extract_headers():
    return ["sofifa_id", "sofifa_name", "sofifa_country"]

def extract_rows_from_html(soup, limit=None):
    league_data = []
    rows = soup.select("table tbody tr")
    
    if limit:
        rows = rows[:limit]
        
    for row in tqdm(rows, desc="Parsing Leagues", unit=" league"):
        try:
            cols = row.find_all("td")
            
            # Bulletproof check
            if len(cols) < 3:
                continue
                
            # 1. ID & NAME EXTRACTION
            name_td = cols[1]
            link = name_td.find("a", href=lambda h: h and "/league/" in h)
            
            league_id = link['href'].split('/')[2] if link else ""
            league_name = link.get_text(strip=True) if link else name_td.get_text(strip=True)
            
            # 2. BULLETPROOF COUNTRY EXTRACTION
            # Scans the entire row for the nation link instead of guessing the column index
            country_name = "Unknown"
            nation_link = row.find("a", href=lambda h: h and "na=" in h)
            
            if nation_link:
                # 1st Priority: Extract from the flag image's title attribute
                img = nation_link.find("img")
                if img and img.has_attr("title"):
                    country_name = img["title"]
                # 2nd Priority: Extract from the anchor tag's title attribute
                elif nation_link.has_attr("title"):
                    country_name = nation_link["title"]
            
            league_data.append([league_id, league_name, country_name])
            
        except Exception as e:
            continue
            
    return league_data

# ==========================================
# PHASE 3: THE TURBO BROWSER THREAD
# ==========================================
def _run_playwright_pipeline(is_test=True):
    with sync_playwright() as p:
        # headless=False is required to pass Cloudflare's bot detection
        browser = p.chromium.launch(headless=False)
        context = browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        page = context.new_page()
        
        print("Launching browser to solve potential Cloudflare challenge...")
        page.goto(BASE_URL)
        
        try:
            page.wait_for_selector("table tbody tr", timeout=30000)
            print("Table loaded successfully.")
        except Exception as e:
            print("Failed to load table in time. Please try again.")
            browser.close()
            return

        # === RESOURCE BLOCKER (TURBO MODE) ===
        def block_heavy_resources(route):
            if route.request.resource_type in ["image", "stylesheet", "font", "media"]:
                route.abort()
            else:
                route.continue_()
                
        context.route("**/*", block_heavy_resources)
        print("Turbo mode activated (Images/CSS blocked).\n")

        # Extract the page source once
        soup = BeautifulSoup(page.content(), "html.parser")
        headers = extract_headers()

        # ==========================================
        # TEST BATCH LOGIC
        # ==========================================
        if is_test:
            print("--- RUNNING 5-LEAGUE VALIDATION TEST ---")
            leagues = extract_rows_from_html(soup, limit=5)

            print(f"\nSuccessfully Fetched! Extracted {len(headers)} Columns.")
            print("-" * 50)
            for i, league in enumerate(leagues):
                league_dict = dict(zip(headers, league))
                print(f"League {i+1}: {league_dict}")
            print("-" * 50)
            print("Test complete. Clear to run production scrape.\n")
            
        # ==========================================
        # PRODUCTION SCRAPE LOGIC
        # ==========================================
        else:
            print("--- STARTING SINGLE-PAGE PRODUCTION SCRAPE ---")
            
            leagues = extract_rows_from_html(soup)
            
            # Write to CSV in one clean operation
            with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(headers)
                writer.writerows(leagues)

            print(f"\nScraping complete! {len(leagues)} leagues safely saved to {CSV_FILENAME}")

        browser.close()

# ==========================================
# EXECUTION CONTROLS
# ==========================================
def run_test():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=True).result()

def run_production():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=False).result()

C:\Users\ashwy\AppData\Local\Temp\ipykernel_19068\2741405667.py:9: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\ashwy\AppData\Local\Temp\ipykernel_19068\2741405667.py:9: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


In [13]:
run_test()

Launching browser to solve potential Cloudflare challenge...
Table loaded successfully.
Turbo mode activated (Images/CSS blocked).

--- RUNNING 5-LEAGUE VALIDATION TEST ---


Parsing Leagues: 100%|██████████| 5/5 [00:00<00:00, 6419.20 league/s]


Successfully Fetched! Extracted 3 Columns.
--------------------------------------------------
League 1: {'sofifa_id': '1', 'sofifa_name': 'Superliga', 'sofifa_country': 'Denmark'}
League 2: {'sofifa_id': '4', 'sofifa_name': 'Pro League', 'sofifa_country': 'Belgium'}
League 3: {'sofifa_id': '7', 'sofifa_name': 'Série A', 'sofifa_country': 'Brazil'}
League 4: {'sofifa_id': '10', 'sofifa_name': 'Eredivisie', 'sofifa_country': 'Netherlands'}
League 5: {'sofifa_id': '13', 'sofifa_name': 'Premier League', 'sofifa_country': 'England'}
--------------------------------------------------
Test complete. Clear to run production scrape.



In [14]:
run_production()

Launching browser to solve potential Cloudflare challenge...
Table loaded successfully.
Turbo mode activated (Images/CSS blocked).

--- STARTING SINGLE-PAGE PRODUCTION SCRAPE ---


Parsing Leagues: 100%|██████████| 52/52 [00:00<00:00, 8702.22 league/s]


Scraping complete! 52 leagues safely saved to data/sofifa/newdata/sofifa_raw_leagues.csv


In [21]:
import os
import re
import pandas as pd
import unicodedata
from rapidfuzz import fuzz

# ==========================================
# 1. CONFIGURATION & SEMANTIC MAPPING
# ==========================================
PATHS = {
    "matched": "newdata/leagues/matched",
    "partial": "newdata/leagues/partial match",
    "unmatched": "newdata/leagues/no match"
}
for path in PATHS.values():
    os.makedirs(path, exist_ok=True)

# Semantic dictionary to resolve geographic database inconsistencies
COUNTRY_MAP = {
    'england': 'united kingdom',
    'scotland': 'united kingdom',
    'wales': 'united kingdom',
    'northern ireland': 'united kingdom',
    'usa': 'united states',
    'korea republic': 'south korea',
    'republic of ireland': 'ireland',
    'china pr': 'china',
    'turkiye': 'turkey' 
}

def normalize_country(country_str):
    """Strips accents from country names and applies semantic mapping."""
    if pd.isna(country_str): return ""
    
    c = str(country_str).lower()
    c = unicodedata.normalize('NFKD', c).encode('ASCII', 'ignore').decode('utf-8')
    c = c.strip()
    
    return COUNTRY_MAP.get(c, c)

def clean_league_name(name):
    """Strips accents, punctuation, and standardizes text for the math engine."""
    if pd.isna(name): return ""
    name = str(name).lower()
    name = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('utf-8')
    name = re.sub(r'[^a-z0-9\s]', '', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

# ==========================================
# 2. LOAD & PREP DATASETS
# ==========================================
master_df = pd.read_csv('data/unified_tables/leagues/matched/final_combined_leagues.csv')
sofifa_df = pd.read_csv('data/sofifa/newdata/sofifa_raw_leagues.csv')
# Enforce the sofifa_ prefix for column clarity downstream
sofifa_df = sofifa_df.rename(columns=lambda x: x if x.startswith('sofifa_') else f'sofifa_{x}')

# Dimensionality Reduction (Squash time-series)
unique_base_leagues = master_df[['soccersolver_id', 'soccersolver_name', 'soccersolver_country']].drop_duplicates()

# ==========================================
# 3. TRIAGE MATCHING ENGINE (DUAL-SCORING)
# ==========================================
potential_matches = []

for _, base_row in unique_base_leagues.iterrows():
    best_score = 0
    best_match_id = None
    
    base_country_norm = normalize_country(base_row['soccersolver_country'])
    base_name_clean = clean_league_name(base_row['soccersolver_name'])
    
    for _, sof_row in sofifa_df.iterrows():
        sof_country_norm = normalize_country(sof_row['sofifa_country'])
        
        # Geographic Anchoring (Fast-Fail)
        if base_country_norm != sof_country_norm:
            continue
            
        sof_name_clean = clean_league_name(sof_row['sofifa_name'])
        
        # DUAL-SCORING ENGINE
        # Score 1: Token Sort (Handles inverted structures: "League Premier" vs "Premier League")
        score_token = fuzz.token_sort_ratio(base_name_clean, sof_name_clean)
        
        # Score 2: Stripped Ratio (Handles spacing inconsistencies: "La Liga 2" vs "LaLiga 2")
        score_stripped = fuzz.ratio(base_name_clean.replace(" ", ""), sof_name_clean.replace(" ", ""))
        
        # Take the absolute best score of the two methods
        score = max(score_token, score_stripped)
        
        if score > best_score:
            best_score = score
            best_match_id = sof_row['sofifa_id']
            
    # Capture everything 40% and above into the global pool
    if best_match_id and best_score >= 40:
        potential_matches.append({
            'soccersolver_id': base_row['soccersolver_id'],
            'sofifa_id': best_match_id,
            'sofifa_match_score': best_score
        })

pool_df = pd.DataFrame(potential_matches)

# ==========================================
# 4. GLOBAL DEDUPLICATION (1-TO-1 ENFORCER)
# ==========================================
if not pool_df.empty:
    pool_df = pool_df.sort_values('sofifa_match_score', ascending=False)
    pool_df = pool_df.drop_duplicates(subset=['soccersolver_id'])
    pool_df = pool_df.drop_duplicates(subset=['sofifa_id'])

# ==========================================
# 5. HISTORICAL EXPLOSION (PRE-ROUTING MERGE)
# ==========================================
# Merge the 1-to-1 dictionary onto the massive time-series master file
exploded_df = master_df.merge(pool_df, on='soccersolver_id', how='left')
# Pull in the rest of the Sofifa columns
final_df = exploded_df.merge(sofifa_df, on='sofifa_id', how='left')

# ==========================================
# 6. ZERO DATA LOSS ROUTING
# ==========================================
# 1. Perfect Matches (> 75)
perfect_df = final_df[final_df['sofifa_match_score'] > 75].copy()

# 2. Partial Matches (40 to 75)
partial_df = final_df[(final_df['sofifa_match_score'] >= 40) & (final_df['sofifa_match_score'] <= 75)].copy()
partial_df['APPROVED'] = '' 

# 3. SoccerSolver Orphans (Failed to hit 40% threshold)
base_orphans = final_df[final_df['sofifa_match_score'].isna()].copy()
sofifa_cols = [c for c in final_df.columns if c.startswith('sofifa_')]
base_orphans.drop(columns=sofifa_cols, errors='ignore', inplace=True)

# 4. Sofifa Orphans (Using index isolation)
matched_sofifa_ids = set(pool_df['sofifa_id'].dropna()) if not pool_df.empty else set()
sofifa_orphans = sofifa_df[~sofifa_df['sofifa_id'].isin(matched_sofifa_ids)].copy()

# ==========================================
# 7. EXPORTS & MATH VERIFICATION
# ==========================================
perfect_df.to_csv(os.path.join(PATHS['matched'], 'leagues_matched.csv'), index=False)
partial_df.to_csv(os.path.join(PATHS['partial'], 'leagues_partial.csv'), index=False)
base_orphans.to_csv(os.path.join(PATHS['unmatched'], 'soccersolver_leagues_unmatched.csv'), index=False)
sofifa_orphans.to_csv(os.path.join(PATHS['unmatched'], 'sofifa_leagues_unmatched.csv'), index=False)

print("\n--- Routing Complete ---")
print(f"- Perfect Matches (Rows): {len(perfect_df)}")
print(f"- Require Manual Review (Rows): {len(partial_df)}")
print(f"- Unmatched SoccerSolver (Rows): {len(base_orphans)}")
print(f"- Unmatched Sofifa (Entities): {len(sofifa_orphans)}")

total_ss_processed = len(perfect_df) + len(partial_df) + len(base_orphans)
print(f"\n[MATH CHECK] Perfect + Partial + Orphans = {total_ss_processed} (Should exactly equal {len(master_df)})")


--- Routing Complete ---
- Perfect Matches (Rows): 162
- Require Manual Review (Rows): 28
- Unmatched SoccerSolver (Rows): 77
- Unmatched Sofifa (Entities): 24

[MATH CHECK] Perfect + Partial + Orphans = 267 (Should exactly equal 267)


In [22]:
# ==========================================
# CONFIGURATION
# ==========================================
PATH_MATCHED = 'newdata/leagues/matched/leagues_matched.csv'
PATH_PARTIAL = 'newdata/leagues/partial match/leagues_partial.csv' 
PATH_FINAL_OUTPUT = 'newdata/leagues/matched/master_leagues_final_integrated.csv'

# ==========================================
# 1. LOAD DATASETS
# ==========================================
try:
    perfect_df = pd.read_csv(PATH_MATCHED)
    partial_df = pd.read_csv(PATH_PARTIAL)
except FileNotFoundError as e:
    print(f"Error: {e}. Ensure you have run the matching script and reviewed the partials.")
    raise

# ==========================================
# 2. ISOLATE APPROVED PARTIALS
# ==========================================
# Safely convert the APPROVED column to numeric, ignoring text or blanks
partial_df['APPROVED'] = pd.to_numeric(partial_df['APPROVED'], errors='coerce')

# Isolate explicitly approved leagues
approved_partials = partial_df[partial_df['APPROVED'] == 1].copy()

# Drop the helper column to align schemas perfectly
approved_partials.drop(columns=['APPROVED'], errors='ignore', inplace=True)

# ==========================================
# 3. INTEGRATION & EXPORT
# ==========================================
# Concatenate vertically
final_master_df = pd.concat([perfect_df, approved_partials], ignore_index=True)

# Export the final master file
final_master_df.to_csv(PATH_FINAL_OUTPUT, index=False)

# ==========================================
# 4. VERIFICATION
# ==========================================
print("--- Post-Review Integration Complete ---")
print(f"Perfect Matches Loaded:    {len(perfect_df)}")
print(f"Approved Partials Added:   {len(approved_partials)}")
print(f"Total Leagues in Master:   {len(final_master_df)}")
print(f"\nFinal dataset saved to: {PATH_FINAL_OUTPUT}")

--- Post-Review Integration Complete ---
Perfect Matches Loaded:    162
Approved Partials Added:   7
Total Leagues in Master:   169

Final dataset saved to: newdata/leagues/matched/master_leagues_final_integrated.csv


In [1]:
import csv
import sys
import time
import random
import asyncio
from bs4 import BeautifulSoup
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from playwright.sync_api import sync_playwright

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ==========================================
# PHASE 1: CONFIGURATION & URL HANDLING
# ==========================================
# REPLACE THIS with your massive URL containing all custom team columns
CUSTOM_URL = "https://sofifa.com/teams?type=club&r=260033&set=true&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps" 

# Season Controls (Leave empty for current season)
FIFA_VERSION = "" 
ROSTER_ID = ""    

CSV_FILENAME = "data/sofifa/newdata/teams-seasonwise/FC26.csv"
TOTAL_TEAMS = 1500
TEAMS_PER_PAGE = 60

# Safely inject season parameters into your custom URL
if FIFA_VERSION and ROSTER_ID:
    separator = "&" if "?" in CUSTOM_URL else "?"
    BASE_URL = f"{CUSTOM_URL}{separator}r={ROSTER_ID}&set={FIFA_VERSION}"
else:
    BASE_URL = CUSTOM_URL

# ==========================================
# PHASE 2: DYNAMIC EXTRACTION LOGIC
# ==========================================
def extract_headers_from_html(soup):
    header_row = soup.select_one("table thead tr")
    if not header_row:
        return []
    
    headers = []
    for th in header_row.find_all("th"):
        text = th.get_text(strip=True)
        
        # Dynamically rename the crest picture column to Team_ID
        if not text and 'col-avatar' in th.get('class', []):
            headers.append("Team_ID")
        elif not text:
            headers.append("Unknown")
        else:
            headers.append(text)
            
    # Add a Season ID column header
    headers.append("Season_Version")
    return headers

def extract_rows_from_html(soup, limit=None):
    team_data = []
    rows = soup.select("table tbody tr")
    
    season_tag = FIFA_VERSION if FIFA_VERSION else "Latest"
        
    for row in rows:
        try:
            cols = row.find_all("td")
            
            # Bulletproof ad blocker
            if len(cols) < 5:
                continue
                
            row_values = []
            
            for td in cols:
                classes = td.get('class', [])
                
                # 1. NAME EXTRACTION
                if any(c in classes for c in ['col-name', 'col-name-wide']):
                    links = td.find_all("a", href=lambda h: h and "/team/" in h)
                    name_text = ""
                    for a in links:
                        if not a.find("img") and a.get_text(strip=True):
                            name_text = a.get_text(strip=True)
                            break
                            
                    if not name_text:
                        name_div = td.select_one(".bp3-text-overflow-ellipsis")
                        name_text = name_div.get_text(strip=True) if name_div else td.get_text(separator=" ", strip=True)
                        
                    row_values.append(name_text)
                        
                # 2. AVATAR/HIDDEN ID EXTRACTION
                elif 'col-avatar' in classes:
                    link = td.find("a", href=lambda h: h and "/team/" in h)
                    row_values.append(link['href'].split('/')[2] if link else "")
                        
                # 3. ALL OTHER CUSTOM STATS
                else:
                    row_values.append(td.get_text(separator=" ", strip=True))
            
            row_values.append(season_tag)
            team_data.append(row_values)
            
            if limit and len(team_data) == limit:
                break
            
        except Exception as e:
            continue
            
    return team_data

# ==========================================
# PHASE 3: THE BROWSER THREAD
# ==========================================
def _run_playwright_pipeline(is_test=True):
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        context = browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        page = context.new_page()
        
        # Safely handle pagination parameters
        separator = "&" if "?" in BASE_URL else "?"
        initial_url = f"{BASE_URL}{separator}offset=0"
        
        print("Launching browser to solve Cloudflare challenge...")
        page.goto(initial_url)
        
        try:
            page.wait_for_selector("table tbody tr", timeout=30000)
            print("Challenge passed! Table loaded.")
        except Exception as e:
            print("Failed to bypass Cloudflare in time. Please try again.")
            browser.close()
            return

        # ==========================================
        # TEST BATCH LOGIC
        # ==========================================
        if is_test:
            print("--- RUNNING 10-TEAM VALIDATION TEST ---")
            
            # Page is already loaded from the challenge step, extract directly
            soup = BeautifulSoup(page.content(), "html.parser")
            columns = extract_headers_from_html(soup)
            teams = extract_rows_from_html(soup, limit=10)

            print(f"Successfully Fetched! Extracted {len(columns)} Columns.")
            print("-" * 50)
            for i, team in enumerate(teams):
                team_dict = dict(zip(columns, team))
                print(f"Team {i+1}: {team_dict}")
            print("-" * 50)
            print("Test complete. Clear to run production scrape.\n")
            
        # ==========================================
        # PRODUCTION SCRAPE LOGIC
        # ==========================================
        else:
            season_display = FIFA_VERSION if FIFA_VERSION else "Latest Version"
            print(f"--- STARTING PRODUCTION SCRAPE (Up to {TOTAL_TEAMS} Teams | Season: {season_display}) ---")
            
            # Extract headers from the already loaded initial page
            soup = BeautifulSoup(page.content(), "html.parser")
            headers = extract_headers_from_html(soup)
            
            with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(headers)
                
            # Master Loop
            with tqdm(total=TOTAL_TEAMS, desc=f"Scraping Season {season_display}", unit=" teams") as pbar:
                for offset in range(0, TOTAL_TEAMS, TEAMS_PER_PAGE):
                    url = f"{BASE_URL}{separator}offset={offset}"
                    
                    for attempt in range(3):
                        try:
                            # For the first loop (offset=0), this reloads the same page, 
                            # but ensures the loop logic stays consistent
                            page.goto(url)
                            page.wait_for_selector("table tbody tr", timeout=30000)
                            
                            soup = BeautifulSoup(page.content(), "html.parser")
                            teams = extract_rows_from_html(soup)
                            
                            with open(CSV_FILENAME, mode="a", newline="", encoding="utf-8") as file:
                                writer = csv.writer(file)
                                writer.writerows(teams)
                            
                            pbar.update(len(teams))
                            
                            if len(teams) == 0:
                                print("\n[Notice] No more teams found. End of database reached.")
                                browser.close()
                                return
                            break
                            
                        except Exception as e:
                            # This will print the actual technical reason it failed
                            print(f"\n[Scrape Error]: {str(e)}")
                            print("Retrying in 10 seconds...")
                            time.sleep(10)
                    
                    time.sleep(random.uniform(2.0, 3.5))

            print(f"\nScraping complete! Data safely saved to {CSV_FILENAME}")

        browser.close()

# ==========================================
# EXECUTION CONTROLS
# ==========================================
def run_team_test():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=True).result()

def run_team_production():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=False).result()

C:\Users\ashwy\AppData\Local\Temp\ipykernel_8420\3601019842.py:12: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\ashwy\AppData\Local\Temp\ipykernel_8420\3601019842.py:12: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


In [2]:
run_team_test()

Launching browser to solve Cloudflare challenge...
Challenge passed! Table loaded.
--- RUNNING 10-TEAM VALIDATION TEST ---
Successfully Fetched! Extracted 12 Columns.
--------------------------------------------------
Team 1: {'Unknown': '', 'Name': 'Paris Saint-Germain Ligue 1', 'ID': '73', 'Overall': '85', 'Attack': '85', 'Midfield': '86', 'Defence': '86', 'Transfer budget': '€232.4M', 'Club worth': '€4B', 'Players': '27', 'Season_Version': 'Latest'}
Team 2: {'Unknown': '', 'Name': 'FC Barcelona La Liga', 'ID': '241', 'Overall': '85', 'Attack': '87', 'Midfield': '85', 'Defence': '83', 'Transfer budget': '€77.5M', 'Club worth': '€4.9B', 'Players': '29', 'Season_Version': 'Latest'}
Team 3: {'Unknown': '', 'Name': 'Real Madrid La Liga', 'ID': '243', 'Overall': '85', 'Attack': '90', 'Midfield': '84', 'Defence': '82', 'Transfer budget': '€146M', 'Club worth': '€5.8B', 'Players': '35', 'Season_Version': 'Latest'}
Team 4: {'Unknown': '', 'Name': 'Arsenal Premier League', 'ID': '1', 'Overall

In [ ]:
run_team_production()

Launching browser to solve Cloudflare challenge...
Challenge passed! Table loaded.
--- STARTING PRODUCTION SCRAPE (Up to 1500 Teams | Season: Latest Version) ---


Scraping Season Latest Version:  29%|██▉       | 438/1500 [04:43<11:00,  1.61 teams/s]


[Scrape Error]: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("table tbody tr") to be visible

Retrying in 10 seconds...


Scraping Season Latest Version:  32%|███▏      | 476/1500 [06:30<21:39,  1.27s/ teams]


[Scrape Error]: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("table tbody tr") to be visible
    - waiting for" https://sofifa.com/teams?type=club&r=260033&set=true&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&offset=720" navigation to finish...
    - navigated to "https://sofifa.com/teams?type=club&r=260033&set=true&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&offset=720"

Retrying in 10 seconds...

[Scrape Error]: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("table tbody tr") to be visible
    - waiting for" https://sofifa.com/teams?type=club&r=260033&set=true&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&offset=720" navigation to finish..

In [ ]:
# --- NEW RESUME VARIABLE ---
# 420 covers teams 421-480. We do this to ensure no dropped records from the crash.
START_OFFSET = 660  

if FIFA_VERSION and ROSTER_ID:
    separator = "&" if "?" in CUSTOM_URL else "?"
    BASE_URL = f"{CUSTOM_URL}{separator}r={ROSTER_ID}&set={FIFA_VERSION}"
else:
    BASE_URL = CUSTOM_URL

# ==========================================
# PHASE 2: DYNAMIC EXTRACTION LOGIC
# ==========================================
def extract_headers_from_html(soup):
    header_row = soup.select_one("table thead tr")
    if not header_row:
        return []
    
    headers = []
    for th in header_row.find_all("th"):
        text = th.get_text(strip=True)
        
        if not text and 'col-avatar' in th.get('class', []):
            headers.append("Team_ID")
        elif not text:
            headers.append("Unknown")
        else:
            headers.append(text)
            
    headers.append("Season_Version")
    return headers

def extract_rows_from_html(soup, limit=None):
    team_data = []
    rows = soup.select("table tbody tr")
    
    season_tag = FIFA_VERSION if FIFA_VERSION else "Latest"
        
    for row in rows:
        try:
            cols = row.find_all("td")
            
            if len(cols) < 5:
                continue
                
            row_values = []
            
            for td in cols:
                classes = td.get('class', [])
                
                if any(c in classes for c in ['col-name', 'col-name-wide']):
                    links = td.find_all("a", href=lambda h: h and "/team/" in h)
                    name_text = ""
                    for a in links:
                        if not a.find("img") and a.get_text(strip=True):
                            name_text = a.get_text(strip=True)
                            break
                            
                    if not name_text:
                        name_div = td.select_one(".bp3-text-overflow-ellipsis")
                        name_text = name_div.get_text(strip=True) if name_div else td.get_text(separator=" ", strip=True)
                        
                    row_values.append(name_text)
                        
                elif 'col-avatar' in classes:
                    link = td.find("a", href=lambda h: h and "/team/" in h)
                    row_values.append(link['href'].split('/')[2] if link else "")
                        
                else:
                    row_values.append(td.get_text(separator=" ", strip=True))
            
            row_values.append(season_tag)
            team_data.append(row_values)
            
            if limit and len(team_data) == limit:
                break
            
        except Exception as e:
            continue
            
    return team_data

# ==========================================
# PHASE 3: THE BROWSER THREAD
# ==========================================
def _run_playwright_pipeline(is_test=False):
    with sync_playwright() as p:
        browser = p.chromium.launch(headless=False)
        context = browser.new_context(
            viewport={"width": 1920, "height": 1080},
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36"
        )
        page = context.new_page()
        
        separator = "&" if "?" in BASE_URL else "?"
        initial_url = f"{BASE_URL}{separator}offset={START_OFFSET}"
        
        print(f"Launching browser to solve Cloudflare challenge at offset {START_OFFSET}...")
        page.goto(initial_url)
        
        try:
            page.wait_for_selector("table tbody tr", timeout=30000)
            print("Challenge passed! Table loaded.")
        except Exception as e:
            print(f"Failed to bypass Cloudflare. Error: {str(e)}")
            browser.close()
            return

        # ==========================================
        # PRODUCTION SCRAPE LOGIC
        # ==========================================
        season_display = FIFA_VERSION if FIFA_VERSION else "Latest Version"
        print(f"--- RESUMING SCRAPE (From offset {START_OFFSET} up to {TOTAL_TEAMS} Teams | Season: {season_display}) ---")
        
        # If starting from 0, write headers. If resuming, skip header writing.
        if START_OFFSET == 0:
            soup = BeautifulSoup(page.content(), "html.parser")
            headers = extract_headers_from_html(soup)
            with open(CSV_FILENAME, mode="w", newline="", encoding="utf-8") as file:
                writer = csv.writer(file)
                writer.writerow(headers)
        else:
            print("Appending to existing CSV to prevent data overwrite...")
            
        # Master Loop
        # tqdm's `initial` parameter visually sets the progress bar to where we left off
        with tqdm(total=TOTAL_TEAMS, initial=START_OFFSET, desc=f"Scraping Season {season_display}", unit=" teams") as pbar:
            for offset in range(START_OFFSET, TOTAL_TEAMS, TEAMS_PER_PAGE):
                url = f"{BASE_URL}{separator}offset={offset}"
                
                for attempt in range(3):
                    try:
                        page.goto(url)
                        page.wait_for_selector("table tbody tr", timeout=30000)
                        
                        soup = BeautifulSoup(page.content(), "html.parser")
                        teams = extract_rows_from_html(soup)
                        
                        # Mode is strictly "a" to append seamlessly
                        with open(CSV_FILENAME, mode="a", newline="", encoding="utf-8") as file:
                            writer = csv.writer(file)
                            writer.writerows(teams)
                        
                        pbar.update(len(teams))
                        
                        if len(teams) == 0:
                            print("\n[Notice] No more teams found. End of database reached.")
                            browser.close()
                            return
                        break
                        
                    except Exception as e:
                        print(f"\n[Scrape Error]: {str(e)}")
                        print("Retrying in 10 seconds...")
                        time.sleep(10)
                
                time.sleep(random.uniform(3.0, 6.0))

        print(f"\nScraping complete! Data safely appended to {CSV_FILENAME}")
        browser.close()

# ==========================================
# EXECUTION CONTROLS
# ==========================================
def run_team_production():
    with ThreadPoolExecutor(max_workers=1) as executor:
        executor.submit(_run_playwright_pipeline, is_test=False).result()

# Execute the resume
run_team_production()

Launching browser to solve Cloudflare challenge at offset 660...
Challenge passed! Table loaded.
--- RESUMING SCRAPE (From offset 660 up to 1500 Teams | Season: Latest Version) ---
Appending to existing CSV to prevent data overwrite...


Scraping Season Latest Version:  44%|████▍     | 663/1500 [00:20<1:36:13,  6.90s/ teams]


[Scrape Error]: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("table tbody tr") to be visible
    - waiting for" https://sofifa.com/teams?type=club&r=260033&set=true&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&offset=720" navigation to finish...
    - navigated to "https://sofifa.com/teams?type=club&r=260033&set=true&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&offset=720"

Retrying in 10 seconds...

[Scrape Error]: Page.wait_for_selector: Timeout 30000ms exceeded.
Call log:
  - waiting for locator("table tbody tr") to be visible
    - waiting for" https://sofifa.com/teams?type=club&r=260033&set=true&showCol%5B%5D=ti&showCol%5B%5D=oa&showCol%5B%5D=at&showCol%5B%5D=md&showCol%5B%5D=df&showCol%5B%5D=tb&showCol%5B%5D=cw&showCol%5B%5D=ps&offset=720" navigation to finish..

In [1]:
#cleaning fc26 data
import re
import sys
import asyncio
import pandas as pd

if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# ==========================================
# PHASE 1: CONFIGURATION & PATHS
# ==========================================
TEAMS_CSV_PATH = "data/sofifa/newdata/teams-seasonwise/RAW/FC26.csv"
LEAGUES_CSV_PATH = "data/sofifa/newdata/sofifa_raw_leagues.csv"  
OUTPUT_CSV_PATH = "data/sofifa/newdata/teams-seasonwise/CLEANED/FC26_cleaned.csv"
# Load Datasets
df_teams = pd.read_csv(TEAMS_CSV_PATH)
df_leagues = pd.read_csv(LEAGUES_CSV_PATH)

print(f"Loaded {len(df_teams)} raw team records.")

# ==========================================
# PHASE 2: COLUMN CLEANING (PURGE UNKNOWN)
# ==========================================
# Drop any column that contains the word 'Unknown' (case-insensitive)
columns_to_keep = [col for col in df_teams.columns if 'unknown' not in col.lower()]
df_teams = df_teams[columns_to_keep]

# Remove leading/trailing whitespaces from the column headers
df_teams.columns = df_teams.columns.str.strip()

# ==========================================
# PHASE 3: DYNAMIC REGEX CONSTRUCTION
# ==========================================
# Clean reference league data based on your specific columns
df_leagues['sofifa_name'] = df_leagues['sofifa_name'].astype(str).str.strip()
df_leagues['sofifa_country'] = df_leagues['sofifa_country'].astype(str).str.strip()

# Create a mapping dictionary for looking up country via league name later
league_to_country_map = dict(zip(df_leagues['sofifa_name'], df_leagues['sofifa_country']))

# Extract unique league names and sort by length descending to protect nested names
# e.g., "2. Bundesliga" or "Bundesliga"
league_list = df_leagues['sofifa_name'].unique()
league_list_sorted = sorted(league_list, key=len, reverse=True)

# Escape special characters (like the dot in "2. Bundesliga") and join with pipe operators
escaped_leagues = [re.escape(league) for league in league_list_sorted]
league_pattern_group = "|".join(escaped_leagues)

# Regex pattern: Captures team name up to the space preceding an authentic league name
regex_pattern = rf"^(.+)\s({league_pattern_group})$"

# ==========================================
# PHASE 4: EXTRACTION & COUNTRY MAPPING
# ==========================================
def split_team_and_league(name_string):
    if pd.isna(name_string):
        return None, None
    
    match = re.match(regex_pattern, str(name_string).strip())
    if match:
        return match.group(1).strip(), match.group(2).strip()
    else:
        return name_string, "Unknown/Unmatched League"

# Apply regex split
extracted_data = df_teams['Name'].apply(lambda x: pd.Series(split_team_and_league(x)))
df_teams['Team_Name'] = extracted_data[0]
df_teams['League_Name'] = extracted_data[1]

# Map country dynamically from our reference league map
df_teams['country'] = df_teams['League_Name'].map(league_to_country_map).fillna("Unknown")

# Check if any teams failed to match an authentic league
unmatched = df_teams[df_teams['League_Name'] == "Unknown/Unmatched League"]
if len(unmatched) > 0:
    print(f"\n[Warning] {len(unmatched)} rows failed to map to an authentic league string.")
    print(unmatched['Name'].head())

# Drop the original composite 'Name' column
df_teams.drop(columns=['Name'], inplace=True)

# Reorder columns to put identity details up front
all_cols = list(df_teams.columns)
identity_cols = ['Team_Name', 'League_Name', 'country']
ordered_cols = identity_cols + [col for col in all_cols if col not in identity_cols]
df_teams = df_teams[ordered_cols]
print(df_teams.head())

C:\Users\ashwy\AppData\Local\Temp\ipykernel_27016\3569732302.py:8: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\ashwy\AppData\Local\Temp\ipykernel_27016\3569732302.py:8: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


Loaded 711 raw team records.
             Team_Name     League_Name  country   ID  Overall  Attack  \
0  Paris Saint-Germain         Ligue 1   France   73       85      85   
1         FC Barcelona         La Liga    Spain  241       85      87   
2          Real Madrid         La Liga    Spain  243       85      90   
3              Arsenal  Premier League  Ukraine    1       84      84   
4            Liverpool  Premier League  Ukraine    9       84      87   

   Midfield  Defence Transfer budget Club worth Players Season_Version  
0        86     86.0         €232.4M        €4B      27         Latest  
1        85     83.0          €77.5M      €4.9B      29         Latest  
2        84     82.0           €146M      €5.8B      35         Latest  
3        85     85.0         €153.3M      €2.5B      23         Latest  
4        84     83.0         €124.4M      €4.7B      30         Latest  


In [2]:
# ==========================================
# PHASE 5: EXPORT CLEANED DATA
# ==========================================
df_teams.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"\nCleaned dataset successfully saved to: {OUTPUT_CSV_PATH}")
print(df_teams[['Team_Name', 'League_Name', 'country']].head(10))


Cleaned dataset successfully saved to: data/sofifa/newdata/teams-seasonwise/CLEANED/FC26_cleaned.csv
             Team_Name     League_Name  country
0  Paris Saint-Germain         Ligue 1   France
1         FC Barcelona         La Liga    Spain
2          Real Madrid         La Liga    Spain
3              Arsenal  Premier League  Ukraine
4            Liverpool  Premier League  Ukraine
5      Manchester City  Premier League  Ukraine
6    FC Bayern München      Bundesliga  Austria
7                Inter         Serie A  Ecuador
8      Atlético Madrid         La Liga    Spain
9          Aston Villa  Premier League  Ukraine


In [11]:
import pandas as pd
import re
from unidecode import unidecode
from IPython.display import display

# --- Configuration ---
FILE_DATASET_1 = 'data/unified_tables/teams/matched/final_combined_teams.csv'
FILE_DATASET_2 = 'data/sofifa/newdata/teams-seasonwise/CLEANED/FC26_cleaned.csv'
TARGET_SEASON = '2025-2026'

# --- The Semantic Alias Dictionary ---
# Keys and values must be strictly lowercase with no punctuation.
TEAM_ALIASES = {
    # Italy
    'inter': 'inter milan',
    'juve': 'juventus',
    'roma': 'as roma',
    'lazio': 'ss lazio',
    
    # England
    'spurs': 'tottenham hotspur',
    'wolves': 'wolverhampton wanderers',
    'man city': 'manchester city',
    'man utd': 'manchester united',
    'manchester utd': 'manchester united',
    'newcastle utd': 'newcastle united',
    'nottm forest': 'nottingham forest',
    'sheff utd': 'sheffield united',
    'sheff wed': 'sheffield wednesday',
    'qpr': 'queens park rangers',
    'brighton': 'brighton hove albion',
    
    # Spain
    'barca': 'barcelona',
    'atleti': 'atletico madrid',
    'sociedad': 'real sociedad',
    'betis': 'real betis',
    
    # France
    'psg': 'paris saint germain',
    'om': 'olympique marseille',
    'ol': 'olympique lyonnais',
    
    # Germany
    'bvb': 'borussia dortmund',
    'gladbach': 'borussia monchengladbach',
    'bayern': 'bayern munich',
    'leverkusen': 'bayer leverkusen',
    'rbl': 'rb leipzig',
    
    # USA / MLS
    'nycfc': 'new york city fc',
    'lafc': 'los angeles fc',
    'lag': 'la galaxy'
}

COUNTRY_ALIASES = {
    'united states': 'usa',
    'united kingdom': 'england',
    'republic of ireland': 'ireland',
    'korea republic': 'korea',
    'korea dpr': 'north korea',
    'netherlands': 'holland',
    'turkiye': 'turkey'
}

In [12]:

def normalize_string(text):
    """
    Standardizes strings for matching: lowercase, removes accents/special chars, 
    and trims excess whitespace.
    """
    if pd.isna(text):
        return ""
    
    # Decode unicode to ascii (e.g., München -> Munchen) and lowercase
    text = unidecode(str(text).lower())
    # Remove all special characters, keeping only alphanumeric and spaces
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # Strip leading/trailing whitespaces and reduce multiple spaces to a single space
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("Loading datasets...")
df1_raw = pd.read_csv(FILE_DATASET_1)
df2_raw = pd.read_csv(FILE_DATASET_2)

print(f"Isolating {TARGET_SEASON} season data...")
# Keep the target season for our matching pool
df1_target = df1_raw[df1_raw['soccersolver_season'] == TARGET_SEASON].copy()

# Park the rest of the seasons to satisfy the "no data deletion" rule
df_other_seasons = df1_raw[df1_raw['soccersolver_season'] != TARGET_SEASON].copy()

print("Adding 'sofifa_' prefix to Dataset 2 columns...")
df2_target = df2_raw.rename(columns=lambda x: f"sofifa_{x}")

print("Normalizing strings and creating match keys...")
# Fallback to wyscout_name if soccersolver_name is missing
df1_names = df1_target['soccersolver_name'].fillna(df1_target['wyscout_name'])

# Apply normalization
df1_target['match_key_name'] = df1_names.apply(normalize_string)
df1_target['match_key_country'] = df1_target['soccersolver_country'].apply(normalize_string)

df2_target['match_key_name'] = df2_target['sofifa_Team_Name'].apply(normalize_string)
df2_target['match_key_country'] = df2_target['sofifa_country'].apply(normalize_string)

# print("Applying semantic aliases for known edge cases...")
# Map the normalized strings through the alias dictionary
print("Applying semantic aliases for teams and countries...")
# Team Aliases
df1_target['match_key_name'] = df1_target['match_key_name'].replace(TEAM_ALIASES)
df2_target['match_key_name'] = df2_target['match_key_name'].replace(TEAM_ALIASES)

# Country Aliases (NEW)
df1_target['match_key_country'] = df1_target['match_key_country'].replace(COUNTRY_ALIASES)
df2_target['match_key_country'] = df2_target['match_key_country'].replace(COUNTRY_ALIASES)

print("Phase 1 Complete!")


Loading datasets...
Isolating 2025-2026 season data...
Adding 'sofifa_' prefix to Dataset 2 columns...
Normalizing strings and creating match keys...
Applying semantic aliases for teams and countries...
Phase 1 Complete!


In [13]:
# print("--- Data Shapes ---")
print(f"Dataset 1 (Target Season Pool): {df1_target.shape}")
print(f"Dataset 1 (Other Seasons Parked): {df_other_seasons.shape}")
print(f"Dataset 2 (SoFIFA Pool): {df2_target.shape}\n")

print("--- Dataset 1 Normalization Preview ---")
display(df1_target[['soccersolver_name', 'match_key_name', 'soccersolver_country', 'match_key_country']].head())

print("\n--- Dataset 2 Normalization Preview ---")
display(df2_target[['sofifa_Team_Name', 'match_key_name', 'sofifa_country', 'match_key_country']].head())


Dataset 1 (Target Season Pool): (1092, 31)
Dataset 1 (Other Seasons Parked): (6589, 29)
Dataset 2 (SoFIFA Pool): (711, 14)

--- Dataset 1 Normalization Preview ---


,soccersolver_name,match_key_name,soccersolver_country,match_key_country
0,Atlético de San Luis,atletico de san luis,Mexico,mexico
6,CF Monterrey,cf monterrey,Mexico,mexico
14,Blackpool FC,blackpool fc,England,england
29,Orlando City SC,orlando city sc,USA,usa
39,Club León FC,club leon fc,Mexico,mexico



--- Dataset 2 Normalization Preview ---


,sofifa_Team_Name,match_key_name,sofifa_country,match_key_country
0,Paris Saint-Germain,paris saintgermain,France,france
1,FC Barcelona,fc barcelona,Spain,spain
2,Real Madrid,real madrid,Spain,spain
3,Arsenal,arsenal,Ukraine,ukraine
4,Liverpool,liverpool,Ukraine,ukraine


In [14]:
import pandas as pd
import re
from rapidfuzz import fuzz, process
from rapidfuzz.distance import JaroWinkler

# --- Helper Function for Tier 3 (Abbreviation Stripping) ---
def strip_abbreviations(name):
    # Strip common football suffixes/prefixes
    pattern = r'\b(fc|cf|sc|ac|afc|ud|cd|rc|de|real|club|utd|united|city)\b'
    stripped = re.sub(pattern, '', name.lower())
    # Clean up any resulting double spaces
    return re.sub(r'\s+', ' ', stripped).strip()

print("Starting Phase 2: Geographic & Structural Triage...")

# Create a master list to hold all viable matches before 1-to-1 deduplication
global_matches = []

# 1. Geographic Anchoring
# Get unique countries present in both datasets
shared_countries = set(df1_target['match_key_country']).intersection(set(df2_target['match_key_country']))

for country in shared_countries:
    # Filter both datasets down to just the current country
    df1_geo = df1_target[df1_target['match_key_country'] == country]
    df2_geo = df2_target[df2_target['match_key_country'] == country]
    
    # Compare every SS/WY team in this country against every SoFIFA team in this country
    for _, ss_row in df1_geo.iterrows():
        ss_name = ss_row['match_key_name']
        
        for _, sf_row in df2_geo.iterrows():
            sf_name = sf_row['match_key_name']
            
            # Initial Scoring Pass
            score = fuzz.token_set_ratio(ss_name, sf_name)
            
            if score >= 75:
                # Compile the match record
                match_record = {
                    'soccersolver_id': ss_row['soccersolver_soccersolver_id'],
                    'sofifa_ID': sf_row['sofifa_ID'],
                    'ss_name_original': ss_row['soccersolver_name'],
                    'sf_name_original': sf_row['sofifa_Team_Name'],
                    'match_key_country': country,
                    'initial_score': score,
                    'status': 'Pending',
                    'reject_reason': ''
                }
                
                # --- THE INSPECTION FUNNEL ---
                
                # Perfect Match Gate
                if score >= 95:
                    match_record['status'] = 'Perfect Match'
                
                # Partial Match Triage (75 - 94)
                else:
                    # Gate 1: Demographic Check (Youth/Gender)
                    wyscout_cat = str(ss_row.get('wyscout_category', '')).lower()
                    wyscout_gen = str(ss_row.get('wyscout_gender', '')).lower()
                    
                    if 'youth' in wyscout_cat or 'female' in wyscout_gen:
                         # Unless SoFIFA explicitly says women/youth (rare), kill it
                         if 'women' not in sf_name and 'u21' not in sf_name and 'u23' not in sf_name:
                             match_record['status'] = 'Rejected'
                             match_record['reject_reason'] = 'Demographic Gate (Youth/Female)'
                    
                    if match_record['status'] != 'Rejected':
                        # Gate 2: Jaro-Winkler Consensus (Derby Shield)
                        jw_score = JaroWinkler.normalized_similarity(ss_name, sf_name) * 100
                        if jw_score < 70: # Substantial suffix divergence
                            match_record['status'] = 'Rejected'
                            match_record['reject_reason'] = f'Jaro-Winkler Penalty ({jw_score:.1f}%)'
                    
                    if match_record['status'] != 'Rejected':
                        # Gate 3: Abbreviation Stripping (Auto-Promotion)
                        ss_stripped = strip_abbreviations(ss_name)
                        sf_stripped = strip_abbreviations(sf_name)
                        stripped_score = fuzz.token_set_ratio(ss_stripped, sf_stripped)
                        
                        if stripped_score >= 95:
                            match_record['status'] = 'Perfect Match'
                        else:
                            match_record['status'] = 'Partial Match'

                # Add to global pool regardless of status (so we can deduplicate properly)
                global_matches.append(match_record)

print("Funnel processing complete.")

Starting Phase 2: Geographic & Structural Triage...
Funnel processing complete.


In [15]:
print("Executing Global 1-to-1 Deduplication...")

# Convert matches to DataFrame
df_matches = pd.DataFrame(global_matches)

# Keep only active matches
df_active_matches = df_matches[df_matches['status'] != 'Rejected'].copy()

# Sort by highest score first
df_active_matches = df_active_matches.sort_values(by='initial_score', ascending=False)

# The 1-to-1 Enforcer: Drop duplicate SS IDs, then drop duplicate SoFIFA IDs
df_active_matches = df_active_matches.drop_duplicates(subset=['soccersolver_id'])
df_active_matches = df_active_matches.drop_duplicates(subset=['sofifa_ID'])

# ... (Previous Phase 2 loop and 1-to-1 Enforcer remain exactly the same) ...

# --- SPLIT THE TIERS ---
df_perfect = df_active_matches[df_active_matches['status'] == 'Perfect Match']
df_partial = df_active_matches[df_active_matches['status'] == 'Partial Match']

print("Squashing SoFIFA dataset to prevent Cartesian explosion...")
# CRITICAL FIX: Reduce SoFIFA to a strict 1-to-1 lookup table
df2_static = df2_target.drop_duplicates(subset=['sofifa_ID'])

print("Merging match keys onto base datasets...")
# Merge 1: Attach the matched SoFIFA ID to the base SoccerSolver dataset
tier1_perfect = pd.merge(df1_target, df_perfect[['soccersolver_id', 'sofifa_ID']], 
                         left_on='soccersolver_soccersolver_id', right_on='soccersolver_id')
# Merge 2: Broadcast the static SoFIFA columns onto the dataset
tier1_perfect = pd.merge(tier1_perfect, df2_static, on='sofifa_ID')

# Repeat for Partial Matches
tier2_partial = pd.merge(df1_target, df_partial[['soccersolver_id', 'sofifa_ID', 'initial_score']], 
                         left_on='soccersolver_soccersolver_id', right_on='soccersolver_id')
tier2_partial = pd.merge(tier2_partial, df2_static, on='sofifa_ID')
tier2_partial['Approved'] = "" # Blank column for manual review

# --- PHASE 3: ISOLATE UNMATCHED CSVs ---
# Find IDs that successfully matched (Perfect + Partial)
matched_ss_ids = df_active_matches['soccersolver_id'].tolist()
matched_sf_ids = df_active_matches['sofifa_ID'].tolist()

# Extract orphans using inverse subsetting against the ORIGINAL target dataframes
unmatched_sswy = df1_target[~df1_target['soccersolver_soccersolver_id'].isin(matched_ss_ids)]
unmatched_sofifa = df2_target[~df2_target['sofifa_ID'].isin(matched_sf_ids)]

# --- EXPORT ---
print("Exporting deduplicated files...")
drop_cols = ['match_key_name_x', 'match_key_country_x', 'match_key_name_y', 'match_key_country_y', 'soccersolver_id']

tier1_perfect.drop(columns=drop_cols, errors='ignore').to_csv('newdata/teams/matched/seasonwise/FC26_perfect_matches.csv', index=False)
tier2_partial.drop(columns=drop_cols, errors='ignore').to_csv('newdata/teams/partial match/FC26_partial_matches_pending.csv', index=False)
unmatched_sswy.drop(columns=['match_key_name', 'match_key_country'], errors='ignore').to_csv('newdata/teams/no match/seasonwise/sswy/unmatched_SSWY_26.csv', index=False)
unmatched_sofifa.drop(columns=['match_key_name', 'match_key_country'], errors='ignore').to_csv('newdata/teams/no match/seasonwise/sofifa/unmatched_sofifa_26.csv', index=False)

print("\n--- Final Match Distribution ---")
print(f"Perfect Matches: {len(tier1_perfect)}")
print(f"Partial Matches (Pending Review): {len(tier2_partial)}")
print(f"Unmatched SS/WY (Dataset 1): {len(unmatched_sswy)}")
print(f"Unmatched SoFIFA (Dataset 2): {len(unmatched_sofifa)}")

Executing Global 1-to-1 Deduplication...
Squashing SoFIFA dataset to prevent Cartesian explosion...
Merging match keys onto base datasets...
Exporting deduplicated files...

--- Final Match Distribution ---
Perfect Matches: 277
Partial Matches (Pending Review): 3
Unmatched SS/WY (Dataset 1): 812
Unmatched SoFIFA (Dataset 2): 294
